# Data

> Loaders for every input, each cached in `data/raw/` after the first download.

In [ ]:
#| default_exp data

In [ ]:
#| export
import json
import zipfile

import geopandas as gpd
import osmnx as ox
import pandas as pd
import requests
import shapely
from shapely.geometry import LineString

from tunnel.config import BNG, COD_DIR, NPS_DIR, PRICE_YEAR, RAW, STUDY_BBOX, WGS84

## OpenStreetMap

In [ ]:
#| export
OVERPASS_URLS = [
    "https://overpass-api.de/api/interpreter",
    "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
]
HEADERS = {"User-Agent": "tunnel-stuff (Rotherhithe crossing research)"}


def overpass(query: str, cache: str) -> list[dict]:
    "Elements returned by an Overpass `query`, cached as `data/raw/<cache>.json`. Tries each mirror in turn."
    path = RAW / f"{cache}.json"
    if not path.exists():
        for url in OVERPASS_URLS:
            r = requests.post(url, data={"data": query}, headers=HEADERS, timeout=180)
            if r.ok:
                break
        r.raise_for_status()
        path.write_text(r.text)
    return json.loads(path.read_text())["elements"]


def ways_to_lines(elements: list[dict]) -> gpd.GeoDataFrame:
    "Overpass ways fetched with `out geom` as BNG lines."
    return gpd.GeoDataFrame(
        {"id": [e["id"] for e in elements], "name": [e.get("tags", {}).get("name", "") for e in elements]},
        geometry=[LineString([(p["lon"], p["lat"]) for p in e["geometry"]]) for e in elements],
        crs=WGS84,
    ).to_crs(BNG)


def street_graph(kind: str):
    "OSM `walk` or `bike` network for the study area."
    path = RAW / f"{kind}.graphml"
    if path.exists():
        return ox.load_graphml(path)
    G = ox.graph_from_bbox(STUDY_BBOX, network_type=kind, simplify=True)
    ox.save_graphml(G, path)
    return G


def _osm_features(cache: str, tags: dict, bbox=STUDY_BBOX) -> gpd.GeoDataFrame:
    path = RAW / f"{cache}.gpkg"
    if path.exists():
        return gpd.read_file(path)
    f = ox.features_from_bbox(bbox, tags=tags).reset_index()
    f = f[["element", "id", "name", "geometry"] + [c for c in tags if c in f.columns]]
    f.to_file(path)
    return f


def stations() -> gpd.GeoDataFrame:
    "Rail, Tube and DLR stations (and piers) in the study area, as points."
    st = _osm_features("stations", {"public_transport": "station"})
    st["geometry"] = st.geometry.representative_point()
    return st


def thames() -> gpd.GeoDataFrame:
    "The River Thames centre line, in BNG."
    return _osm_features("thames", {"waterway": "river"}).query("name == 'River Thames'").to_crs(BNG)


def thames_line():
    "The Thames centre line as one merged line."
    return shapely.line_merge(thames().union_all())


def jubilee_tunnels() -> gpd.GeoDataFrame:
    "Jubilee line tunnel ways near the crossing, in BNG."
    q = (
        '[out:json][timeout:60];way["railway"="subway"]["tunnel"="yes"]["name"~"Jubilee"]'
        "(51.495,-0.050,51.510,-0.010);out tags geom;"
    )
    return ways_to_lines(overpass(q, "jubilee_tunnels_osm"))


def open_space(bbox) -> gpd.GeoDataFrame:
    "Parks, water, schools and places of worship inside `bbox` (lon/lat), in BNG, with a `class` column."
    path = RAW / "constraints_osm.gpkg"
    if path.exists():
        return gpd.read_file(path).to_crs(BNG)
    tags = {
        "leisure": ["park", "nature_reserve", "playground", "pitch", "garden", "common"],
        "landuse": ["recreation_ground", "cemetery", "grass", "village_green"],
        "amenity": ["school", "place_of_worship"],
        "natural": ["water"],
        "water": True,
    }
    f = ox.features_from_bbox(bbox, tags=tags)
    f = f[f.geometry.type.isin(["Polygon", "MultiPolygon"])].copy()
    is_water = (f.get("natural") == "water") | f.get("water").notna()
    is_comm = f.get("amenity").isin(["school", "place_of_worship"])
    f["class"] = "open_space"
    f.loc[is_comm, "class"] = "community"
    f.loc[is_water, "class"] = "water"
    f = f[["class", "name", "geometry"]].reset_index(drop=True)
    f.to_file(path)
    return f.to_crs(BNG)

## ONS: boundaries and Census 2021

In [ ]:
#| export
ONS_ARCGIS = "https://services1.arcgis.com/ESMARspQHYMw9BZ9/arcgis/rest/services/{}/FeatureServer/0/query"
OA_CENTROIDS = "OA_December_2021_EW_PWC_V4"  # population-weighted centroids
OA_BOUNDARIES = "Output_Areas_2021_EW_BGC_V2"
MSOA_BOUNDARIES = "Middle_layer_Super_Output_Areas_December_2021_Boundaries_EW_BGC_V3"


def ons_layer(service: str, bbox=STUDY_BBOX) -> gpd.GeoDataFrame:
    "ONS Open Geography features intersecting `bbox`, in WGS84."
    path = RAW / f"{service}.gpkg"
    if path.exists():
        return gpd.read_file(path)
    frames, offset = [], 0
    while True:
        r = requests.get(
            ONS_ARCGIS.format(service),
            timeout=120,
            params={
                "where": "1=1",
                "outFields": "*",
                "f": "geojson",
                "outSR": 4326,
                "geometry": ",".join(map(str, bbox)),
                "geometryType": "esriGeometryEnvelope",
                "inSR": 4326,
                "spatialRel": "esriSpatialRelIntersects",
                "resultOffset": offset,
                "resultRecordCount": 2000,
            },
        )
        r.raise_for_status()
        if not r.json().get("features"):
            break
        gdf = gpd.read_file(r.text)
        frames.append(gdf)
        offset += len(gdf)
    out = gpd.GeoDataFrame(pd.concat(frames, ignore_index=True), crs=WGS84)
    out.to_file(path, driver="GPKG")
    return out


TS044_URL = "https://www.nomisweb.co.uk/output/census/2021/census2021-ts044.zip"


def households() -> pd.DataFrame:
    "Census 2021 households by output area (table TS044)."
    path = RAW / "ts044.zip"
    if not path.exists():
        r = requests.get(TS044_URL, timeout=300)
        r.raise_for_status()
        path.write_bytes(r.content)
    with zipfile.ZipFile(path) as z, z.open("census2021-ts044-oa.csv") as f:
        hh = pd.read_csv(f, usecols=["geography code", "Accommodation type: Total: All households"])
    hh.columns = ["OA21CD", "households"]
    return hh

## House prices

HM Land Registry Price Paid data by postcode district, geocoded to postcode centroids with postcodes.io.

In [ ]:
#| export
PPD_URL = "https://landregistry.data.gov.uk/app/ppd/ppd_data.csv"


def price_paid(district: str, since: str) -> pd.DataFrame:
    "Every Price Paid record in a postcode district since `since` (YYYY-MM-DD)."
    path = RAW / f"ppd_{district}_{since}.csv"
    if not path.exists():
        r = requests.get(
            PPD_URL,
            params={"header": "true", "limit": "all", "postcode": district, "min_date": since},
            timeout=600,
        )
        r.raise_for_status()
        path.write_bytes(r.content)
    return pd.read_csv(path, dtype=str)


def geocode(postcodes: list[str]) -> pd.DataFrame:
    "Postcode centroids (`postcode`, `lon`, `lat`), cached in `data/raw/postcodes.csv`."
    path = RAW / "postcodes.csv"
    cached = pd.read_csv(path) if path.exists() else pd.DataFrame(columns=["postcode", "lon", "lat"])
    todo = sorted(set(postcodes) - set(cached["postcode"]))
    rows = []
    for i in range(0, len(todo), 100):
        r = requests.post("https://api.postcodes.io/postcodes", json={"postcodes": todo[i : i + 100]}, timeout=60)
        r.raise_for_status()
        for item in r.json()["result"]:
            res = item["result"]
            rows.append(
                {
                    "postcode": item["query"],
                    "lon": res["longitude"] if res else None,
                    "lat": res["latitude"] if res else None,
                }
            )
    if rows:
        cached = pd.concat([cached, pd.DataFrame(rows)], ignore_index=True)
        cached.to_csv(path, index=False)
    return cached

## Prices and land values

In [ ]:
#| export
DEFLATOR_URL = "https://www.ons.gov.uk/generator?format=csv&uri=/economy/grossdomesticproductgdp/timeseries/ybgb/qna"


def gdp_deflator() -> pd.Series:
    "ONS GDP deflator (YBGB), annual, indexed by year."
    path = RAW / "gdp_deflator.csv"
    if not path.exists():
        r = requests.get(DEFLATOR_URL, headers=HEADERS, timeout=60)
        r.raise_for_status()
        path.write_bytes(r.content)
    d = pd.read_csv(path, skiprows=8, header=None, names=["period", "index"])
    d = d[d.period.str.fullmatch(r"\d{4}")].astype({"period": int, "index": float})
    return d.set_index("period")["index"]


def real(gbp: float, year: int) -> float:
    "Convert `gbp` in `year` prices to `PRICE_YEAR` prices."
    defl = gdp_deflator()
    return gbp * defl[PRICE_YEAR] / defl[year]


def land_values() -> pd.DataFrame:
    """MHCLG residential land value estimates for policy appraisal (2023), one row per local authority.

    Download the workbook from gov.uk ("Land value estimates for policy appraisal 2023") to
    `data/raw/land_values_2023.xlsx`.
    """
    lv = pd.read_excel(RAW / "land_values_2023.xlsx", sheet_name="Residential", header=None, skiprows=5)
    lv = lv.iloc[:, 1:22]
    lv.columns = ["region", "la_code", "la", "dph_low", "dph_med", "dph_high"] + [
        f"{d}_{p}" for d in ("low", "med", "high") for p in (25, 40, 50, 60, 75)
    ]
    return lv.dropna(subset=["la"]).set_index("la")

## Land Registry: title plans and owners

The National Polygon Service (title plans) and CCOD/OCOD (UK and overseas company owners) need an
HM Land Registry account: see `NPS_DIR` and `COD_DIR` in the config.

In [ ]:
#| export
def title_plans(area) -> gpd.GeoDataFrame:
    "NPS title plan polygons that intersect `area` (a BNG geometry)."
    files = sorted(NPS_DIR.glob("LR_POLY_FULL_*_[0-9].shp"))
    if not files:
        raise FileNotFoundError(f"No NPS shapefiles in {NPS_DIR}: set NPS_DIR")
    parts = []
    for shp in files:
        part = gpd.read_file(shp, bbox=area.bounds, engine="pyogrio")
        parts.append(part[part.intersects(area)])
    g = gpd.GeoDataFrame(pd.concat(parts, ignore_index=True), crs=BNG).drop_duplicates("POLY_ID")
    g.columns = [c.lower() if c.isupper() else c for c in g.columns]
    g["geometry"] = shapely.make_valid(g.geometry.values)
    return g.sort_values("poly_id").reset_index(drop=True)


COD_COLS = {
    "Title Number": "title_no",
    "Tenure": "tenure",
    "Property Address": "address",
    "Proprietor Name (1)": "owner",
    "Company Registration No. (1)": "company_no",
    "Proprietorship Category (1)": "owner_category",
}


def company_owners(titles: set[str]) -> pd.DataFrame:
    "CCOD/OCOD rows for `titles`. Titles not in either file are owned by private individuals."
    files = sorted(COD_DIR.glob("[CO]COD_FULL_*.zip"))
    if not files:
        raise FileNotFoundError(f"No CCOD/OCOD zips in {COD_DIR}: set COD_DIR")
    frames = []
    for path in files:
        for chunk in pd.read_csv(path, usecols=list(COD_COLS), dtype=str, chunksize=500_000):
            frames.append(chunk[chunk["Title Number"].isin(titles)].rename(columns=COD_COLS))
    return pd.concat(frames, ignore_index=True).drop_duplicates("title_no")


def tenures(titles: set[str]) -> pd.Series:
    "Tenure (`Freehold`/`Leasehold`) of any title, from the NPS tenure file, indexed by title number."
    path = next(NPS_DIR.glob("LR_TENURE_FULL_*.csv"))
    frames = []
    for chunk in pd.read_csv(
        path, header=None, usecols=[0, 2], names=["title_no", "code"], dtype=str, chunksize=2_000_000
    ):
        frames.append(chunk[chunk.title_no.isin(titles)])
    t = pd.concat(frames).drop_duplicates("title_no").set_index("title_no").code
    return t.str[-1].map({"F": "Freehold", "L": "Leasehold"}).rename("tenure")

## Check

In [ ]:
print(
    f"{len(stations())} stations, Thames {thames_line().length / 1000:.1f} km, {len(jubilee_tunnels())} Jubilee ways"
)
print(f"£100m in 2017 = £{real(100, 2017):.0f}m in {PRICE_YEAR}")

54 stations, Thames 13.6 km, 10 Jubilee ways
£100m in 2017 = £133m in 2025
